# 03.Train 7-Emotion Classifier

This notebook trains a transformer-based emotion classification model.

## Model

We use:

```text
distilroberta-base
```

This model is smaller and faster than full RoBERTa, while still being strong for text classification.

## Output

The trained model will be saved to:

```text
models/emotion_model_v2
```

In [1]:
# Import required libraries
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score, classification_report

c:\Users\Taruni\Desktop\My Projects\Pet_Pal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Set paths and configuration

In [ ]:
# Project root assumes this notebook is inside: project/notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "emotion_model_v2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train.csv"
VAL_PATH = PROCESSED_DIR / "val.csv"
LABEL_MAPPING_PATH = PROCESSED_DIR / "label_mapping.json"

MODEL_NAME = "distilroberta-base"
MAX_LENGTH = 128

# Training settings
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5

# Keep this True if training on CPU or low-end hardware.
# It limits samples per class for faster training.
QUICK_TRAIN_MODE = False
MAX_SAMPLES_PER_CLASS = 1500

SEED = 42

print("Project root:", PROJECT_ROOT)
print("Model output directory:", MODEL_DIR)

Project root: c:\Users\Taruni\Desktop\PetChat-2.0\emotion_classifier
Model output directory: c:\Users\Taruni\Desktop\PetChat-2.0\emotion_classifier\models\emotion_model_v2


## 2. Set random seed

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


## 3. Load processed data and label mapping

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

with open(LABEL_MAPPING_PATH, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

label2id = label_mapping["label2id"]
id2label = {int(k): v for k, v in label_mapping["id2label"].items()}
UI_LABEL_NAMES = label_mapping["ui_label_names"]

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Labels:", UI_LABEL_NAMES)

display(train_df.head())

Train shape: (14000, 3)
Validation shape: (20781, 3)
Labels: ['happy', 'calm', 'sad', 'angry', 'anxious', 'stressed', 'confused']


,text,label,label_id
0,*pokes head in from Detroit* Hi! *waves!*,calm,1
1,"Yes, you should speak with an attorney.",calm,1
2,That’s a terrible amount of bp for 11 kills,anxious,4
3,"""LEFT TURN YIELD ON GREEN""",calm,1
4,If they want to attack [RELIGION] they should ...,anxious,4


## 4. Optional quick training subset

In [5]:
print(train_df.columns.tolist())
print(val_df.columns.tolist())

['text', 'label', 'label_id']
['text', 'label', 'label_id']


In [6]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")

train_df.columns = train_df.columns.str.strip()
val_df.columns = val_df.columns.str.strip()

if QUICK_TRAIN_MODE:
    train_df = train_df.groupby("label", group_keys=False).sample(
        n=MAX_SAMPLES_PER_CLASS,
        replace=True,
        random_state=SEED
    ).reset_index(drop=True)

    val_df = val_df.groupby("label", group_keys=False).sample(
        n=max(100, MAX_SAMPLES_PER_CLASS // 10),
        replace=True,
        random_state=SEED
    ).reset_index(drop=True)

display(train_df["label"].value_counts().reindex(UI_LABEL_NAMES).to_frame("train_count"))
display(val_df["label"].value_counts().reindex(UI_LABEL_NAMES).to_frame("val_count"))

,train_count
label,
happy,1000
calm,1000
sad,1000
angry,1000
anxious,1000
stressed,1000
confused,1000


,val_count
label,
happy,100
calm,100
sad,100
angry,100
anxious,100
stressed,100
confused,100


## 5. Create Hugging Face datasets

In [7]:
# Keep only the columns needed for training
train_dataset = Dataset.from_pandas(train_df[["text", "label_id"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label_id"]])

# Rename label_id to labels because Hugging Face Trainer expects the column name "labels"
train_dataset = train_dataset.rename_column("label_id", "labels")
val_dataset = val_dataset.rename_column("label_id", "labels")

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['text', 'labels'],
    num_rows: 7000
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 700
})


## 6. Tokenize text

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    # Truncation keeps text within MAX_LENGTH tokens.
    # Padding is handled dynamically by DataCollatorWithPadding.
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_dataset[0])

Map: 100%|██████████| 700/700 [00:00<00:00, 7329.70 examples/s]

{'text': "Why in the world did you marry her? She's ragging on your own sister???", 'labels': 3, 'input_ids': [0, 7608, 11, 5, 232, 222, 47, 12908, 69, 116, 264, 18, 910, 12771, 15, 110, 308, 2761, 38713, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## 7. Load model

In [9]:
num_labels = len(UI_LABEL_NAMES)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

print("Model loaded:", MODEL_NAME)
print("Number of labels:", num_labels)

Loading weights: 100%|██████████| 101/101 [00:00<00:00, 3025.89it/s]
RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilroberta-base
Number of labels: 7


## 8. Define evaluation metrics

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    weighted_f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

## 9. Configure training

In [12]:
training_args = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "training_outputs"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    logging_steps=50,

    disable_tqdm=True,
    report_to="none",

    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer ready.")

Trainer ready.


## 10. Train the model

In [13]:
# Start training
train_result = trainer.train()

print("Training finished.")
print(train_result)

c:\Users\Taruni\Desktop\My Projects\Pet_Pal\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': '1.951', 'grad_norm': '5.355', 'learning_rate': '1.944e-05', 'epoch': '0.05714'}
{'loss': '1.934', 'grad_norm': '5.782', 'learning_rate': '1.887e-05', 'epoch': '0.1143'}
{'loss': '1.765', 'grad_norm': '19.99', 'learning_rate': '1.83e-05', 'epoch': '0.1714'}
{'loss': '1.664', 'grad_norm': '19.65', 'learning_rate': '1.773e-05', 'epoch': '0.2286'}
{'loss': '1.606', 'grad_norm': '12.14', 'learning_rate': '1.715e-05', 'epoch': '0.2857'}
{'loss': '1.564', 'grad_norm': '13.33', 'learning_rate': '1.658e-05', 'epoch': '0.3429'}
{'loss': '1.449', 'grad_norm': '22.94', 'learning_rate': '1.601e-05', 'epoch': '0.4'}
{'loss': '1.376', 'grad_norm': '19.91', 'learning_rate': '1.544e-05', 'epoch': '0.4571'}
{'loss': '1.38', 'grad_norm': '18.38', 'learning_rate': '1.487e-05', 'epoch': '0.5143'}
{'loss': '1.31', 'grad_norm': '20.07', 'learning_rate': '1.43e-05', 'epoch': '0.5714'}
{'loss': '1.472', 'grad_norm': '19.03', 'learning_rate': '1.373e-05', 'epoch': '0.6286'}
{'loss': '1.289', 'grad_nor

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]
c:\Users\Taruni\Desktop\My Projects\Pet_Pal\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': '1.285', 'grad_norm': '16.23', 'learning_rate': '9.726e-06', 'epoch': '1.029'}
{'loss': '1.041', 'grad_norm': '16.57', 'learning_rate': '9.154e-06', 'epoch': '1.086'}
{'loss': '1.101', 'grad_norm': '19.84', 'learning_rate': '8.583e-06', 'epoch': '1.143'}
{'loss': '1.055', 'grad_norm': '24.08', 'learning_rate': '8.011e-06', 'epoch': '1.2'}
{'loss': '1.147', 'grad_norm': '14.37', 'learning_rate': '7.44e-06', 'epoch': '1.257'}
{'loss': '1.147', 'grad_norm': '18.75', 'learning_rate': '6.869e-06', 'epoch': '1.314'}
{'loss': '1.034', 'grad_norm': '14.54', 'learning_rate': '6.297e-06', 'epoch': '1.371'}
{'loss': '1.009', 'grad_norm': '29.11', 'learning_rate': '5.726e-06', 'epoch': '1.429'}
{'loss': '1.043', 'grad_norm': '18.07', 'learning_rate': '5.154e-06', 'epoch': '1.486'}
{'loss': '1.095', 'grad_norm': '21.7', 'learning_rate': '4.583e-06', 'epoch': '1.543'}
{'loss': '1.032', 'grad_norm': '19.68', 'learning_rate': '4.011e-06', 'epoch': '1.6'}
{'loss': '0.9912', 'grad_norm': '35.66

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.53it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

{'train_runtime': '1409', 'train_samples_per_second': '9.934', 'train_steps_per_second': '1.242', 'train_loss': '1.277', 'epoch': '2'}
Training finished.
TrainOutput(global_step=1750, training_loss=1.2770452423095704, metrics={'train_runtime': 1409.2762, 'train_samples_per_second': 9.934, 'train_steps_per_second': 1.242, 'train_loss': 1.2770452423095704, 'epoch': 2.0})


## 11. Evaluate on validation set

In [16]:
val_results = trainer.evaluate()

print("Validation results:")
print(val_results)

{'eval_loss': '1.406', 'eval_accuracy': '0.5171', 'eval_macro_f1': '0.5146', 'eval_weighted_f1': '0.5146', 'eval_runtime': '13.18', 'eval_samples_per_second': '53.1', 'eval_steps_per_second': '6.676', 'epoch': '2'}
Validation results:
{'eval_loss': 1.4056185483932495, 'eval_accuracy': 0.5171428571428571, 'eval_macro_f1': 0.5145854692652049, 'eval_weighted_f1': 0.514585469265205, 'eval_runtime': 13.1819, 'eval_samples_per_second': 53.103, 'eval_steps_per_second': 6.676, 'epoch': 2.0}


## 12. Save final model

In [17]:
# Save model and tokenizer
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

# Save label mapping inside model folder as well
with open(MODEL_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=4)

print("Model saved to:", MODEL_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s]

Model saved to: c:\Users\Taruni\Desktop\PetChat-2.0\emotion_classifier\models\emotion_model_v2


## Training Summary

The model has been trained and saved.

Next, run:

```text
04_evaluate_emotion_model.ipynb
```

That notebook tests the model on the unseen test set and gives the final result.